# Hands-on Projects (Speech AI)

**Module:** 16 — Speech AI

Transcribe & summarize, local Whisper CLI, cascaded bot sim, softphone webhook.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Implement four speech projects with acceptance tests
- Simulate mic loops and webhooks without paid keys
- Document latency, WER proxy, and retention choices


## Project 1 — Transcribe & Summarize

**Brief:** Audio file → transcript → bullet summary + action items JSON.

```mermaid
flowchart LR
  A[Audio] --> S[ASR] --> T[Transcript]
  T --> L[LLM summarize] --> J[JSON]
```

| ID | Acceptance |
|----|------------|
| T1 | Empty audio fails gracefully |
| T2 | Summary has <= 5 bullets |
| T3 | Action items is a list |


In [ ]:
# Project 1 starter
import json

def transcribe_mock(path: str) -> str:
    if not path: raise ValueError("empty path")
    return "Alice: Need the deck by Friday. Bob: I will send it tomorrow."

def summarize(transcript: str) -> dict:
    bullets = [ln.strip() for ln in transcript.split(".") if ln.strip()][:5]
    actions = [{"owner": "Bob", "task": "send deck", "due": "tomorrow"}]
    return {"bullets": bullets, "action_items": actions}

tr = transcribe_mock("call.wav")
summary = summarize(tr)
assert summary["action_items"] and len(summary["bullets"]) <= 5
print(json.dumps(summary, indent=2))


### Try it yourself — Transcribe & summarize

1. Add speaker labels parsing from 'Name:' prefixes.
2. Redact phone numbers in transcripts.
3. Store audio hash not raw bytes in the job record.


## Project 2 — Local Whisper CLI

**Brief:** CLI that takes a wav path and prints text; swapable model size; timing stats.

| ID | Acceptance |
|----|------------|
| W1 | `--input` required |
| W2 | Prints seconds + chars |
| W3 | Nonexistent file exit code != 0 |


In [ ]:
# Project 2 starter (mock whisper)
import time, sys
from pathlib import Path

def whisper_cli(argv):
    args = dict(zip(argv[::2], argv[1::2]))
    path = args.get("--input")
    if not path:
        return 2, "missing --input"
    if not Path(path).exists():
        # create tiny fake for demo OR return error — acceptance wants error for missing
        return 1, "file not found"
    t0 = time.time()
    text = "mock transcript"
    dt = time.time() - t0
    return 0, f"{text}\nsecs={dt:.3f} chars={len(text)} model={args.get('--model','base')}"

# create file then run
Path("demo_tone.wav").write_bytes(b"RIFF..")
code, out = whisper_cli(["--input", "demo_tone.wav", "--model", "tiny"])
print(code, out)
print(whisper_cli(["--input", "missing.wav"])[0])


### Try it yourself — Whisper CLI

1. Add `--language` flag.
2. Write tests for W1–W3.
3. Document when to pick tiny vs small vs large.


## Project 3 — Cascaded Voice Bot (mic loop simulation)

**Brief:** Simulate mic → ASR partials → LLM → TTS text chunks with barge-in.

| ID | Acceptance |
|----|------------|
| B1 | Barge-in clears TTS buffer |
| B2 | Risky tool requires confirm |
| B3 | Session remembers order_id |


In [ ]:
# Project 3 starter
class Bot:
    def __init__(self):
        self.slots = {}
        self.tts = ""
        self.speaking = False
    def asr_final(self, text):
        if "cancel" in text:
            self.slots["intent"] = "cancel_order"
        for tok in text.split():
            if tok.isdigit():
                self.slots["order_id"] = tok
        if self.slots.get("intent") == "cancel_order" and not self.slots.get("confirmed"):
            self.tts = f"Confirm cancel {self.slots.get('order_id','unknown')}?"
        elif self.slots.get("confirmed"):
            self.tts = f"Canceled {self.slots.get('order_id')}"
        self.speaking = True
    def user_yes(self):
        self.slots["confirmed"] = True
        self.asr_final("yes")
    def barge_in(self):
        self.speaking = False
        self.tts = ""

bot = Bot()
bot.asr_final("cancel order 4455")
assert "Confirm" in bot.tts
bot.barge_in()
assert bot.tts == "" and bot.speaking is False
bot.asr_final("cancel order 4455")
bot.user_yes()
assert "Canceled" in bot.tts and bot.slots["order_id"] == "4455"
print("P3 ok", bot.slots)


### Try it yourself — Cascaded bot

1. Add silence timeout reprompt.
2. Integrate PartialStabilizer from notebook 02.
3. Log TTFA-like timestamps per turn.


## Project 4 — Softphone Webhook

**Brief:** HTTP handler for inbound call + call.completed; idempotent; returns start_agent actions.

| ID | Acceptance |
|----|------------|
| H1 | Unknown signature rejected (stub) |
| H2 | Duplicate event_id ignored |
| H3 | completed writes CRM note dict |


In [ ]:
# Project 4 starter
SEEN = set()

def verify(sig, secret="YOUR_WEBHOOK_SECRET"):
    return sig == f"sign:{secret}"

def handle(headers, body):
    if not verify(headers.get("X-Sig", "")):
        return {"status": 401, "error": "bad_sig"}
    eid = body["event_id"]
    if eid in SEEN:
        return {"status": 200, "dup": True}
    SEEN.add(eid)
    if body["type"] == "inbound":
        return {"status": 200, "actions": [{"start_agent": "support_v2"}]}
    if body["type"] == "completed":
        note = {"call_id": body["call_id"], "transcript": body.get("transcript", "")}
        return {"status": 200, "crm": note}
    return {"status": 400}

print(handle({"X-Sig": "nope"}, {"event_id": "1", "type": "inbound"}))
print(handle({"X-Sig": "sign:YOUR_WEBHOOK_SECRET"}, {"event_id": "1", "type": "inbound"}))
print(handle({"X-Sig": "sign:YOUR_WEBHOOK_SECRET"}, {"event_id": "1", "type": "inbound"}))
print(handle({"X-Sig": "sign:YOUR_WEBHOOK_SECRET"}, {"event_id": "2", "type": "completed", "call_id": "c9", "transcript": "hi"}))


## Acceptance Checklist

- [ ] Architecture diagram per project
- [ ] Latency budget for Project 3
- [ ] Retention note (audio vs transcript)
- [ ] >=5 automated asserts total
- [ ] Placeholder secrets only (`YOUR_*`)
- [ ] Failure modes list (WER, barge-in, webhook retries)


In [ ]:
# Shared smokes
assert summarize(transcribe_mock("x"))["action_items"]
assert whisper_cli(["--input", "missing.wav"])[0] != 0
assert handle({"X-Sig": "sign:YOUR_WEBHOOK_SECRET"}, {"event_id": "z", "type": "inbound"})["status"] == 200
print("all speech project smokes passed")


### Try it yourself — Ship

1. Add OpenAPI sketch for the webhook.
2. Load-test idempotency with 100 duplicate events.
3. Wire Project 1 to live ASR with YOUR_OPENAI_API_KEY.

**Stretch:** Record a 10s sample and run local Whisper if installed.


## Knowledge Check

**Q1.** Why simulate barge-in in tests?

<details><summary>Answer</summary>

It's the #1 UX failure in voice bots and rarely caught by text-only unit tests.

</details>

**Q2.** Why webhook signatures?

<details><summary>Answer</summary>

Prevent attackers from forging inbound call events into your agent.

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `TTFA` | Time to first audio |
| `webhook` | HTTP callback from platform |
| `idempotency` | Safe retries of the same event |
| `cascade` | ASR→LLM→TTS agent path |
| `acceptance` | Binary project requirement |


## Key Takeaways

- Projects should exercise ASR, cascade control, and telephony glue
- Test barge-in and idempotency explicitly
- Keep secrets as YOUR_* placeholders
- Document latency and retention with the code


## Production Incident Patterns — speech projects

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Users talk over bot | No barge-in / bad VAD | Tune endpointing; cancel TTS |
| High WER in field | Noise/codec mismatch | Denoise; match sample rate |
| Creepy voice clone | Weak consent policy | Watermark + allow-list |
| 800ms+ dead air | Cascaded STT→LLM→TTS | Speculative TTS; S2S; stream |
| Compliance scare | Raw audio retention | TTL + transcript-only default |

```
Voice control loop:
  mic -> VAD -> ASR partials -> NLU/LLM -> TTS stream -> speaker
                ^                | tools/HITL
                +-- transcripts/metrics/audit --+
```


In [ ]:
# Cross-cutting: never log raw secrets or full audio bytes
import hashlib, json

def audio_audit(user_id: str, wav_bytes: bytes, meta: dict) -> dict:
    return {
        "user_id": user_id,
        "sha256_16": hashlib.sha256(wav_bytes).hexdigest()[:16],
        "nbytes": len(wav_bytes),
        "meta": {k: v for k, v in meta.items() if k not in {"api_key", "authorization"}},
        "topic": "speech projects",
    }

print(json.dumps(audio_audit("u1", b"RIFF....", {"model": "whisper", "api_key": "YOUR_OPENAI_API_KEY"})))


## Mini Case Study — speech projects

**Scenario:** A support org replaces IVR menus with a voice agent. Pilot NPS soars.
**Month 2:** Accents under-served; callers interrupted mid-sentence; recordings retained 2 years.

**Retro questions**
1. What was the latency budget (ASR+LLM+TTS)?
2. Was barge-in tested with noisy headsets?
3. Retention: audio vs transcript vs redacted entities?
4. Which intents require human transfer?

**Design rule:** conversational voice is a real-time distributed system — optimize the path, not only model quality.


In [ ]:
# Cross-cutting: latency budget checker
from dataclasses import dataclass

@dataclass
class VoiceBudget:
    asr_ms: int = 300
    llm_first_token_ms: int = 400
    tts_first_audio_ms: int = 200
    network_ms: int = 100
    def total(self): return self.asr_ms + self.llm_first_token_ms + self.tts_first_audio_ms + self.network_ms
    def ok(self, sla=900): return self.total() <= sla

b = VoiceBudget()
print("speech projects", "total_ms", b.total(), "ok", b.ok())
print("tight", VoiceBudget(500, 600, 300, 150).ok())


### Try it yourself — speech projects ops

1. Draft an on-call runbook bullet list for speech projects when p95 turn latency > SLA.
2. Sketch metrics: WER proxy, barge-in rate, transfer rate, audio retention age.
